In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

import warnings
warnings.filterwarnings("ignore")

# 第 4 课：3D 视觉：从几何基础到神经渲染

本章节将带领大家从经典的相机几何学出发，逐步过渡到现代的神经表示技术，揭开 3D 重建与渲染的神秘面纱。

## 1. 相机几何与坐标变换 (下钻)

### 1.1 针孔相机模型与内参分解
针孔相机模型是 3D 投影的基础。3D 点 $(X, Y, Z)$ 投影到图像平面 $(u, v)$ 的方程由**内参矩阵 $K$** 决定：

$$K = \begin{bmatrix} f_x & s & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

- **$f_x, f_y$**: 像素焦距，定义了图像的尺度。
- **$c_x, c_y$**: 主点坐标，通常是图像的几何中心。
- **$s$**: 倾斜因子 (Skew)，描述像素单元格的倾斜程度（现代相机通常为 0）。

### 1.2 坐标转换链
从 3D 世界到位图的变换可以拆解为以下链条：
1.  **世界坐标 ($P_w$) → 相机坐标 ($P_c$):** 
    $$P_c = R P_w + t$$
    其中 $R, t$ 构成了相机的**外参**。
2.  **相机坐标 ($P_c$) → 归一化平面:** 
    $$[x', y', 1]^T = [X_c/Z_c, Y_c/Z_c, 1]^T$$
3.  **归一化平面 → 像素坐标 ($u, v$):** 
    $$\begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = K \begin{bmatrix} x' \\ y' \\ 1 \end{bmatrix}$$

### 1.3 畸变模型 (Distortion Model)
真实镜头并非完美，通常需要在投影前引入畸变校正：
- **径向畸变 (Radial):** $x_{distorted} = x(1 + k_1r^2 + k_2r^4 + k_3r^6)$
- **切向畸变 (Tangential):** $x_{distorted} = x + [2p_1xy + p_2(r^2 + 2x^2)]$
其中 $r^2 = x^2 + y^2$。

## 2. 传统 3D 重建与几何约束

### 2.1 对极几何 (Epipolar Geometry)
两台相机观察同一个 3D 点时，存在约束关系。由**本质矩阵 $E$** 刻画：
$$x_2^T E x_1 = 0, \quad E = [t]_{\times} R$$
其中 $[t]_{\times}$ 是平移向量的反对称矩阵。对于未标定相机，使用**基础矩阵 $F$**: $F = K_2^{-T} E K_1^{-1}$。

### 2.2 SfM (Structure from Motion) 流程
从一组图像中恢复 3D 结构和相机位姿的典型流程：
1.  **特征提取与匹配**: 如 SIFT, ORB 或深度学习特征 (SuperPoint)。
2.  **特征匹配**: 构建多视图之间的对应关系。
3.  **对极几何计算**: 恢复相对 R, t。
4.  **三角测量 (Triangulation)**: 根据对应点计算 3D 空间坐标。
5.  **BA 优化 (Bundle Adjustment)**: 同时优化相机位姿和 3D 点坐标，最小化重投影误差。
    $$\min_{P, X} \sum_{i,j} \|x_{i,j} - \text{project}(P_i, X_j)\|^2$$

## 3. 神经辐射场 (NeRF) 深度探索

### 3.1 核心原理
NeRF 通过 MLP 学习一个连续的场景函数 $f(x, d) \to (c, \sigma)$。

### 3.2 体渲染离散化实现
实际计算时，沿射线采样 $N$ 个点，将积分转化为求和：
$$\hat{C}(r) = \sum_{i=1}^N w_i c_i, \quad w_i = T_i (1 - \exp(-\sigma_i \delta_i))$$
其中 $T_i = \exp(-\sum_{j=1}^{i-1} \sigma_j \delta_j)$ 是累积透射率。

### 3.3 位置编码 (Positional Encoding)
**下钻**：为什么需要 PE？
神经网络倾向于学习低频函数（频谱偏差）。PE 通过 $\sin/\cos$ 将低维坐标变换为高维，使得 MLP 能够拟合场景中的高频细节（边缘、纹理）。

## 4. 3D Gaussian Splatting (3DGS)

### 4.1 基本单元：3D 高斯椭球体
每个粒子由位置 $\mu$ 和协方差 $\Sigma$ 定义。为了保证正定性，协方差矩阵分解为：
$$\Sigma = R S S^T R^T$$
其中 $R$ 为四元数表示的旋转，$S$ 为缩放向量。

### 4.2 Tiled Rasterization 加速原理
3DGS 将图像划分为 16x16 的 tiles：
1.  对每个 tile，只选择与其相交的高斯粒子。
2.  按深度对这些粒子排序。
3.  并行执行 $\alpha$-blending 渲染。这种方式极大地提高了吞吐量，实现了 100+ FPS。

## 5. 渲染实战：从 3D 点云到 2D 投影

### 5.1 光栅化 vs 光线追踪
- **光栅化 (Rasterization)**: 将 3D 几何体映射到像素（3DGS 方案）。速度快，适合实时交互。
- **光线追踪 (Ray Tracing)**: 模拟光线传播路径（NeRF 方案）。精度高，能自然处理阴影和折射，但计算量大。

In [ ]:
class DifferentiableProjector(nn.Module):
    """
    使用 PyTorch 实现一个可微的投影层
    """
    def __init__(self, fx, fy, cx, cy):
        super().__init__()
        self.register_buffer('K', torch.tensor([
            [fx, 0, cx],
            [0, fy, cy],
            [0, 0, 1]
        ]).float())

    def forward(self, points_3d, R, t):
        """
        points_3d: (N, 3)
        R: (3, 3)
        t: (3,)
        """
        # 1. 外参变换
        points_cam = (torch.matmul(R, points_3d.t()) + t.view(3, 1)).t()
        
        # 2. 透视除法
        z = points_cam[:, 2:] + 1e-6
        points_norm = points_cam[:, :2] / z
        
        # 3. 内参投影
        u = self.K[0, 0] * points_norm[:, 0] + self.K[0, 2]
        v = self.K[1, 1] * points_norm[:, 1] + self.K[1, 2]
        
        return torch.stack([u, v], dim=1)

# 演示可微性 (Gradient Flow)
points = torch.randn(10, 3, requires_grad=True)
R = torch.eye(3)
t = torch.zeros(3)
projector = DifferentiableProjector(fx=500, fy=500, cx=320, cy=240)

uv = projector(points, R, t)
loss = uv.sum()
loss.backward()
print("梯度流动成功! 3D 点梯度样本:\n", points.grad[0])